# COSC726 · Lab 7 — Crew versus Baseline
### Real model · self-contained · four briefs, four different outcomes

**Week 8 · ~2.5 hours · Colab or local**

This lab is not "build a crew" — that is thirty lines and you will have it
working in twenty minutes. The question is **whether the crew was worth
building**, and the 2026 evidence says the burden of proof is on you.

Reported industry testing puts multi-agent at **3–10× the tokens** of a
single agent for equivalent tasks. At *equal* token budgets, a single agent
matches or beats multi-agent on reasoning work.

| Part | You build | Kind |
|---|---|---|
| 1 | The corpus, the briefs, the role contracts | given |
| 2 | **The single-agent baseline — first** | **Task 1** |
| 3 | The crew: researcher → analyst → writer → critic | **Task 2** |
| 4 | Measure both, on four briefs | **Task 3** |
| 5 | Five exercises on real failures | assessed |

### Build the baseline first

Not second. A team that builds the crew first will rationalise it
afterwards — every year. Without a baseline you have no experiment, only a
demo.

### The three failure families

A 2026 study of five multi-agent frameworks over 150+ tasks catalogued 14
failure modes in three families, and concluded that **many are structural**
— not fixable with better prompts.

| **Specification** | roles or the task badly defined — the largest category |
| **Inter-agent** | context lost at a handoff, contradiction, sycophancy |
| **Verification** | nobody checked, or the crew could not stop |

Exercises 2–4 produce one from each. You are asked to **classify** what you
see, not merely report it.

---
## Part 0 — Setup

In [ ]:

# @title Setup — install, credentials, and lab kit on Colab  { display-mode: "form" }
!pip -q install "openai>=1.40" "pydantic>=2.7" 2>&1 | tail -1

import os
import platform
import subprocess
import time
import urllib.request
import urllib.error
import json
import shutil


# ============================================================
# 1. Install Python dependencies
# ============================================================

print("=== 1. Installing Python dependencies ===")

subprocess.run(
    ["pip", "install", "-q", "-U", "chromadb"],
    check=True
)

print("✓ ChromaDB installed")


# ============================================================
# 2. Detect Colab architecture
# ============================================================

print("\n=== 2. Detecting system architecture ===")

machine = platform.machine().lower()

print("Detected architecture:", machine)

if machine in ("x86_64", "amd64"):
    ollama_arch = "amd64"

elif machine in ("aarch64", "arm64"):
    ollama_arch = "arm64"

else:
    raise RuntimeError(
        f"Unsupported architecture: {machine}"
    )

print("✓ Using Ollama architecture:", ollama_arch)


# ============================================================
# 3. Install system dependency: zstd
# ============================================================

print("\n=== 3. Installing zstd ===")

subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True
)

subprocess.run(
    ["apt-get", "install", "-y", "-qq", "zstd"],
    check=True
)

print("✓ zstd installed")


# ============================================================
# 4. Remove broken previous Ollama installation
# ============================================================

print("\n=== 4. Cleaning previous Ollama installation ===")

possible_paths = [
    "/usr/local/bin/ollama",
    "/usr/bin/ollama"
]

for path in possible_paths:
    if os.path.isfile(path):
        try:
            result = subprocess.run(
                [path, "--version"],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                timeout=5
            )

            if result.returncode != 0:
                print("Removing broken Ollama:", path)
                os.remove(path)

        except (OSError, subprocess.SubprocessError):
            print("Removing invalid Ollama:", path)
            os.remove(path)


# Remove previous Ollama libraries if present
if os.path.isdir("/usr/lib/ollama"):
    print("Removing previous Ollama libraries...")
    shutil.rmtree(
        "/usr/lib/ollama",
        ignore_errors=True
    )


# ============================================================
# 5. Download official Ollama Linux archive
# ============================================================

print("\n=== 5. Downloading Ollama ===")

OLLAMA_DOWNLOAD = (
    f"https://ollama.com/download/"
    f"ollama-linux-{ollama_arch}.tar.zst"
)

ARCHIVE_PATH = f"/tmp/ollama-linux-{ollama_arch}.tar.zst"

print("Download URL:")
print(OLLAMA_DOWNLOAD)

download_result = subprocess.run(
    [
        "curl",
        "-fL",
        "--retry", "3",
        "--retry-delay", "2",
        "-o", ARCHIVE_PATH,
        OLLAMA_DOWNLOAD
    ]
)

if download_result.returncode != 0:
    raise RuntimeError(
        "Failed to download Ollama archive."
    )

if not os.path.exists(ARCHIVE_PATH):
    raise RuntimeError(
        "Ollama archive was not downloaded."
    )

archive_size = os.path.getsize(ARCHIVE_PATH)

print(
    "✓ Downloaded:",
    round(archive_size / (1024**3), 2),
    "GB"
)


# ============================================================
# 6. Extract Ollama into /usr
# ============================================================

print("\n=== 6. Extracting Ollama ===")

extract_result = subprocess.run(
    [
        "tar",
        "--zstd",
        "-xf",
        ARCHIVE_PATH,
        "-C",
        "/usr"
    ]
)

if extract_result.returncode != 0:
    raise RuntimeError(
        "Failed to extract Ollama."
    )

print("✓ Ollama extracted")


# ============================================================
# 7. Find Ollama executable
# ============================================================

ollama_path = shutil.which("ollama")

if ollama_path is None:

    candidates = [
        "/usr/bin/ollama",
        "/usr/local/bin/ollama"
    ]

    for candidate in candidates:
        if os.path.exists(candidate):
            ollama_path = candidate
            break


if ollama_path is None:
    raise RuntimeError(
        "Ollama executable could not be found."
    )


print("\nOllama executable:", ollama_path)


# ============================================================
# 8. Verify Ollama executable
# ============================================================

print("\n=== 7. Verifying Ollama ===")

try:

    version = subprocess.run(
        [ollama_path, "--version"],
        capture_output=True,
        text=True,
        timeout=15
    )

except OSError as e:

    raise RuntimeError(
        f"Ollama executable exists but cannot run: {e}"
    )


print(
    version.stdout.strip()
    or version.stderr.strip()
)


if version.returncode != 0:
    raise RuntimeError(
        "Ollama executable failed verification."
    )


print("✓ Ollama binary works")


# ============================================================
# 9. Helper to check Ollama API
# ============================================================

OLLAMA_URL = "http://127.0.0.1:11434"


def ollama_up():

    try:

        with urllib.request.urlopen(
            OLLAMA_URL,
            timeout=2
        ) as response:

            return response.status == 200

    except Exception:
        return False


# ============================================================
# 10. Start Ollama server
# ============================================================

print("\n=== 8. Starting Ollama server ===")


if not ollama_up():

    log_path = "/tmp/ollama.log"

    log_file = open(
        log_path,
        "w"
    )

    env = os.environ.copy()

    # Important for Colab
    env["OLLAMA_HOST"] = "127.0.0.1:11434"

    ollama_process = subprocess.Popen(
        [ollama_path, "serve"],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env=env
    )

    print("Waiting for Ollama API...")

    for i in range(60):

        if ollama_up():
            break

        if ollama_process.poll() is not None:

            log_file.close()

            print("\n--- Ollama log ---")

            if os.path.exists(log_path):

                with open(log_path) as f:
                    print(f.read())

            raise RuntimeError(
                "Ollama server stopped unexpectedly."
            )

        time.sleep(1)


if not ollama_up():

    print("\n--- Ollama log ---")

    if os.path.exists("/tmp/ollama.log"):

        with open("/tmp/ollama.log") as f:
            print(f.read())

    raise RuntimeError(
        "Ollama API did not start."
    )


print("✓ Ollama server running")
print("✓ API:", OLLAMA_URL)


# ============================================================
# 11. Pull qwen2.5:7b model
# ============================================================

MODEL_NAME = "qwen2.5:7b"

print(
    f"\n=== 9. Pulling {MODEL_NAME} ==="
)


pull_result = subprocess.run(
    [
        ollama_path,
        "pull",
        MODEL_NAME
    ]
)


if pull_result.returncode != 0:

    raise RuntimeError(
        f"Failed to pull {MODEL_NAME}"
    )


print(
    f"✓ {MODEL_NAME} ready"
)


# ============================================================
# 12. Show installed models
# ============================================================

print("\n=== 10. Installed models ===")

subprocess.run(
    [
        ollama_path,
        "list"
    ],
    check=False
)




# ============================================================
# 15. Final summary
# ============================================================

print("\n" + "=" * 60)
print("LAB 6 SETUP COMPLETE")
print("=" * 60)

print(
    "Architecture       :",
    ollama_arch
)

print(
    "Ollama executable :",
    ollama_path
)

print(
    "Ollama API        :",
    OLLAMA_URL
)

print(
    " model   :",
    MODEL_NAME
)


print("=" * 60)

=== 1. Installing Python dependencies ===
✓ ChromaDB installed

=== 2. Detecting system architecture ===
Detected architecture: x86_64
✓ Using Ollama architecture: amd64

=== 3. Installing zstd ===
✓ zstd installed

=== 4. Cleaning previous Ollama installation ===

=== 5. Downloading Ollama ===
Download URL:
https://ollama.com/download/ollama-linux-amd64.tar.zst
✓ Downloaded: 1.34 GB

=== 6. Extracting Ollama ===
✓ Ollama extracted

Ollama executable: /usr/bin/ollama

=== 7. Verifying Ollama ===
✓ Ollama binary works

=== 8. Starting Ollama server ===
Waiting for Ollama API...
✓ Ollama server running
✓ API: http://127.0.0.1:11434

=== 9. Pulling qwen2.5:7b ===
✓ qwen2.5:7b ready

=== 10. Installed models ===

LAB 6 SETUP COMPLETE
Architecture       : amd64
Ollama executable : /usr/bin/ollama
Ollama API        : http://127.0.0.1:11434
 model   : qwen2.5:7b


In [ ]:
# @title Write lab8_kit.py into the runtime  { display-mode: "form" }
kit_source = r'''"""
COSC726 Lab 7 — multi-agent systems (support module)
====================================================
Real model, no mocks, no other lab required.

    pip install openai pydantic
    ollama pull qwen2.5:7b && ollama serve
    python layla_crew_solution.py

What this week actually asks
----------------------------
Not "build a crew" -- that is thirty lines. The question is whether the crew
was worth building, and the 2026 evidence says the burden of proof is on
you. Reported figures: multi-agent implementations typically use 3-10x the
tokens of a single agent for equivalent tasks, and at EQUAL token budgets a
single agent matches or beats multi-agent on reasoning work.

So this kit is built to measure, not to impress. Everything is instrumented
per agent: tokens, wall-clock, and what each role actually contributed.

Public API
----------
    BRIEFS              four research briefs with gold claim sets
    Role, RoleSpec      the three workers plus a critic
    Message             one inter-agent message, logged
    CrewTrace           per-agent tokens, latency, handoffs, verdicts
    single_agent()      the baseline you must beat
    score_output()      grounded claims, unsupported claims, coverage
    compare()           the ledger: crew vs baseline
    CORPUS, search()    a small closed corpus, so failures are yours

The failure taxonomy this lab exercises
---------------------------------------
A 2026 study of five multi-agent frameworks over 150+ tasks catalogued 14
failure modes in three families, and concluded that many are STRUCTURAL --
not fixable by better prompts. The three families:

    1. SPECIFICATION      the roles or the task were badly defined
                          (reported as the largest single category)
    2. INTER-AGENT        misalignment: context lost at a handoff, one agent
                          contradicting another, sycophantic agreement
    3. VERIFICATION       nobody checked, or the crew could not stop

Exercises 2-4 produce one from each family. You are asked to classify what
you observe, not merely to report it.
"""
from __future__ import annotations

import json
import os
import re
import time
from dataclasses import dataclass, field
from enum import Enum
from typing import Any

from pydantic import BaseModel, ConfigDict, Field

__all__ = [
    "CORPUS", "search", "BRIEFS", "Brief", "Role", "RoleSpec", "ROLES",
    "Message", "AgentRun", "CrewTrace", "Finding", "Analysis", "Memo",
    "Verdict", "score_output", "compare", "make_client", "MODEL", "PROVIDER",
    "SYSTEMS",
]

PROVIDER = os.getenv("LLM_PROVIDER", "ollama")
MODEL = (os.getenv("OLLAMA_MODEL", "qwen2.5:7b") if PROVIDER == "ollama"
         else os.getenv("OPENAI_MODEL", "gpt-4o-mini-2024-07-18"))


def make_client():
    """The Week 2 seam. One function; nothing else knows the provider."""
    from openai import OpenAI
    if PROVIDER == "ollama":
        base = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
        return OpenAI(base_url=f"{base}/v1", api_key="ollama")
    if not os.getenv("OPENAI_API_KEY"):
        raise SystemExit("OPENAI_API_KEY not set (or use LLM_PROVIDER=ollama)")
    return OpenAI()


# ---------------------------------------------------------------------------
# 1. The corpus — small and closed, so every failure is one you caused
# ---------------------------------------------------------------------------

CORPUS: dict[str, str] = {
    "SUP-2401": (
        "Supplier review, Northwind Retail, Q1. Delta Logistics delivered "
        "94% of consignments on time in Q1, down from 97% in Q4. The decline "
        "is concentrated in the northern depot. Delta's contract renews in "
        "September and includes a 2% liquidated-damages clause per week of "
        "delay, capped at 10%."),
    "SUP-2402": (
        "Supplier review, Northwind Retail, Q2. Delta Logistics delivered "
        "89% on time in Q2. Northwind raised two formal claims under the "
        "damages clause, totalling 4% of consignment value. Delta attributes "
        "the decline to a depot relocation completed in June."),
    "FIN-1180": (
        "Cost note. Late-delivery credits paid to customers rose from "
        "£4,100 in Q1 to £9,700 in Q2. The credit policy threshold is three "
        "working days. Approximately 71% of Q2 credits related to "
        "consignments routed through the northern depot."),
    "OPS-0907": (
        "Operations note. The northern depot relocation ran from April to "
        "June. Throughput during the move was roughly 60% of normal. "
        "Contingency routing through the southern depot was available but "
        "was used for only 12% of affected consignments."),
    "LEG-0455": (
        "Legal note. The liquidated-damages clause in the Delta contract is "
        "the sole remedy for late supply; Northwind cannot additionally "
        "claim consequential losses such as customer credits. Terminating "
        "for convenience requires ninety days' notice."),
}


def search(query: str, k: int = 3) -> list[dict[str, str]]:
    """Lexical retrieval over the corpus. Deliberately simple: retrieval is
    not this week's subject, and a fancy retriever would hide the coordination
    failures we are here to see."""
    q = set(re.findall(r"[a-z0-9]+", query.lower()))
    scored = []
    for doc_id, text in CORPUS.items():
        words = set(re.findall(r"[a-z0-9]+", text.lower()))
        overlap = len(q & words) / (len(q) or 1)
        scored.append((overlap, doc_id, text))
    scored.sort(reverse=True)
    return [{"doc_id": d, "text": t} for s, d, t in scored[:k] if s > 0]


# ---------------------------------------------------------------------------
# 2. The briefs
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class Brief:
    brief_id: str
    question: str
    gold_docs: set[str]
    gold_claims: list[str]      # substrings that must appear, grounded
    trap: str


BRIEFS: list[Brief] = [
    Brief("B1",
          "Why did customer late-delivery credits rise in Q2, and what can "
          "we recover from the supplier?",
          {"FIN-1180", "SUP-2402", "OPS-0907", "LEG-0455"},
          ["northern depot", "relocation", "sole remedy"],
          "The answer spans four documents and the LEGAL constraint reverses "
          "the obvious conclusion: you cannot recover the credits. A crew "
          "that splits research from analysis often loses LEG-0455 at the "
          "handoff \u2014 inter-agent misalignment, and the commonest MAST "
          "family after specification."),

    Brief("B2",
          "Summarise Delta Logistics' on-time performance across Q1 and Q2.",
          {"SUP-2401", "SUP-2402"},
          ["94", "89"],
          "Two documents, two numbers, no reasoning. A single agent should "
          "match or beat the crew here, and the token ratio is the point: "
          "this is the case that shows coordination costing more than it "
          "buys."),

    Brief("B3",
          "Should we terminate the Delta contract?",
          {"SUP-2402", "LEG-0455", "OPS-0907"},
          ["ninety days", "relocation"],
          "There is no correct answer, only a defensible one \u2014 and the "
          "evidence cuts both ways: Delta's decline has a stated cause and "
          "termination needs ninety days' notice. Watch for SYCOPHANCY: the "
          "analyst agreeing with whatever the researcher framed first."),

    Brief("B4",
          "What was Delta's Q3 on-time performance?",
          set(),
          ["not available", "no data", "insufficient"],
          "The corpus contains no Q3 data at all. The correct output says "
          "so. A crew under pressure to produce a memo will often produce "
          "one anyway \u2014 verification failure, the third MAST family."),
]


# ---------------------------------------------------------------------------
# 3. Roles and their contracts
# ---------------------------------------------------------------------------

class Role(str, Enum):
    RESEARCHER = "researcher"
    ANALYST = "analyst"
    WRITER = "writer"
    CRITIC = "critic"


class Finding(BaseModel):
    """What the researcher hands on. Note `doc_id`: a claim without a source
    cannot survive a handoff intact, and losing the source is how a crew
    fabricates."""
    model_config = ConfigDict(extra="forbid")
    doc_id: str
    claim: str = Field(max_length=300)


class Research(BaseModel):
    model_config = ConfigDict(extra="forbid")
    findings: list[Finding] = Field(min_length=1, max_length=10)
    gaps: list[str] = Field(default_factory=list, max_length=5)


class Analysis(BaseModel):
    model_config = ConfigDict(extra="forbid")
    conclusion: str = Field(max_length=600)
    supported_by: list[str] = Field(default_factory=list, max_length=10)
    caveats: list[str] = Field(default_factory=list, max_length=5)


class Memo(BaseModel):
    model_config = ConfigDict(extra="forbid")
    text: str = Field(max_length=1200)
    citations: list[str] = Field(default_factory=list, max_length=10)


class Verdict(BaseModel):
    """The critic's output. `unsupported` is the field that matters: a claim
    with no citation is the failure this whole week is about."""
    model_config = ConfigDict(extra="forbid")
    approved: bool
    unsupported: list[str] = Field(default_factory=list, max_length=6)
    missing: list[str] = Field(default_factory=list, max_length=6)


@dataclass(frozen=True)
class RoleSpec:
    role: Role
    contract: type[BaseModel]
    system: str


SYSTEMS: dict[Role, str] = {
    Role.RESEARCHER: """You are the RESEARCHER on a small analyst team.

Your only job is to retrieve evidence and report it faithfully. You do not
draw conclusions and you do not write prose for a reader.

Use search(query) results only. Every finding MUST carry the doc_id it came
from. If the corpus does not answer part of the question, say so in `gaps`
rather than inferring.

Return ONE JSON object:
{"findings": [{"doc_id": "FIN-1180", "claim": "..."}], "gaps": ["..."]}""",

    Role.ANALYST: """You are the ANALYST. You receive the researcher's
findings and nothing else.

Reason over what you were given. Do NOT introduce facts that are not in the
findings; if a conclusion needs something you were not given, put that in
`caveats` instead of assuming it.

Note especially any finding that CONSTRAINS the obvious conclusion. An
analysis that ignores a constraint is worse than no analysis.

Return ONE JSON object:
{"conclusion": "...", "supported_by": ["FIN-1180"], "caveats": ["..."]}""",

    Role.WRITER: """You are the WRITER. You receive the analyst's conclusion
and the researcher's findings.

Produce a short internal memo, at most six sentences. Every factual claim
must carry a doc_id in `citations`. If the analysis says the evidence is
insufficient, the memo says so plainly \u2014 do not write around a gap.

Return ONE JSON object:
{"text": "...", "citations": ["FIN-1180", "LEG-0455"]}""",

    Role.CRITIC: """You are the CRITIC. You did not write any of this and
you are not trying to be encouraging.

You receive the brief, the findings, and the memo. Check two things:

1. Is every factual claim in the memo traceable to a doc_id in the findings?
   List any that are not in `unsupported`.
2. Does the memo address every part of the brief? List what is missing.

Approve only if `unsupported` is empty. A fluent memo that cites nothing is
the failure you exist to catch.

Return ONE JSON object:
{"approved": false, "unsupported": ["..."], "missing": ["..."]}""",
}

ROLES: dict[Role, RoleSpec] = {
    Role.RESEARCHER: RoleSpec(Role.RESEARCHER, Research,
                              SYSTEMS[Role.RESEARCHER]),
    Role.ANALYST: RoleSpec(Role.ANALYST, Analysis, SYSTEMS[Role.ANALYST]),
    Role.WRITER: RoleSpec(Role.WRITER, Memo, SYSTEMS[Role.WRITER]),
    Role.CRITIC: RoleSpec(Role.CRITIC, Verdict, SYSTEMS[Role.CRITIC]),
}

SINGLE_AGENT_SYSTEM = """You are an analyst at Northwind Retail.

Answer the brief using search(query) results only. Every factual claim must
carry the doc_id it came from. If the corpus does not answer the question,
say so plainly rather than inferring.

Note any evidence that CONSTRAINS the obvious conclusion.

Return ONE JSON object:
{"text": "...", "citations": ["FIN-1180"]}"""


# ---------------------------------------------------------------------------
# 4. Instrumentation — the deliverable
# ---------------------------------------------------------------------------

@dataclass
class Message:
    """One inter-agent handoff, logged. Context loss happens HERE."""
    frm: str
    to: str
    payload: str
    chars: int = 0

    def __post_init__(self):
        self.chars = len(self.payload)


@dataclass
class AgentRun:
    role: str
    tokens: int = 0
    seconds: float = 0.0
    retries: int = 0
    ok: bool = True
    error: str | None = None


@dataclass
class CrewTrace:
    brief_id: str
    topology: str                      # "single" | "sequential" | "critic"
    agents: list[AgentRun] = field(default_factory=list)
    messages: list[Message] = field(default_factory=list)
    memo: Memo | None = None
    verdict: Verdict | None = None
    stop_reason: str = ""

    @property
    def tokens(self) -> int:
        return sum(a.tokens for a in self.agents)

    @property
    def seconds(self) -> float:
        return sum(a.seconds for a in self.agents)

    @property
    def handoff_chars(self) -> int:
        return sum(m.chars for m in self.messages)

    def render(self) -> str:
        out = [f"{self.brief_id}  [{self.topology}]"]
        for a in self.agents:
            flag = "ok" if a.ok else f"ERR {a.error}"
            out.append(f"   {a.role:<12} {a.tokens:>6} tok  "
                       f"{a.seconds:>5.1f}s  retries={a.retries}  {flag}")
        for m in self.messages:
            out.append(f"   {m.frm} -> {m.to}: {m.chars} chars")
        if self.verdict:
            out.append(f"   critic: approved={self.verdict.approved}"
                       f" unsupported={len(self.verdict.unsupported)}")
        out.append(f"   total {self.tokens} tok  {self.seconds:.1f}s  "
                   f"stop: {self.stop_reason}")
        return "\n".join(out)


# ---------------------------------------------------------------------------
# 5. Scoring
# ---------------------------------------------------------------------------

def score_output(brief: Brief, memo: Memo | None) -> dict[str, Any]:
    """Grounding and coverage, measured separately.

    A memo can be beautifully grounded and answer the wrong question, or
    answer perfectly while citing nothing. Report both.
    """
    if memo is None:
        return {"coverage": 0.0, "grounded": 0.0, "cited_docs": [],
                "hallucinated_docs": [], "answered": False}
    low = memo.text.lower()
    hit = sum(1 for c in brief.gold_claims if c.lower() in low)
    cited = [c for c in memo.citations]
    bad = [c for c in cited if c not in CORPUS]
    right = [c for c in cited if c in brief.gold_docs]
    return {"coverage": hit / max(len(brief.gold_claims), 1),
            "grounded": len(right) / max(len(cited), 1) if cited else 0.0,
            "cited_docs": cited,
            "hallucinated_docs": bad,
            "answered": bool(memo.text.strip())}


def compare(rows: list[tuple[str, CrewTrace, dict]]) -> str:
    """The ledger. Token RATIO is the number the week turns on."""
    hdr = (f"{'brief':<7}{'topology':<12}{'tokens':>8}{'secs':>7}"
           f"{'coverage':>10}{'grounded':>10}  notes")
    out = [hdr, "-" * (len(hdr) + 12)]
    base: dict[str, int] = {}
    for label, tr, sc in rows:
        if tr.topology == "single":
            base[tr.brief_id] = tr.tokens
        ratio = ""
        if tr.topology != "single" and base.get(tr.brief_id):
            ratio = f"{tr.tokens / base[tr.brief_id]:.1f}\u00d7 baseline"
        if sc["hallucinated_docs"]:
            ratio += f"  HALLUCINATED {sc['hallucinated_docs']}"
        out.append(f"{tr.brief_id:<7}{tr.topology:<12}{tr.tokens:>8}"
                   f"{tr.seconds:>7.1f}{sc['coverage']:>9.0%}"
                   f"{sc['grounded']:>10.0%}  {ratio}")
    out.append("")
    out.append("Reported industry figures put multi-agent at 3-10\u00d7 the tokens")
    out.append("of a single agent for equivalent work. What did YOU measure,")
    out.append("and did the extra spend buy anything on the coverage column?")
    return "\n".join(out)
'''

with open("lab8_kit.py", "w", encoding="utf-8") as f:
    f.write(kit_source)

import importlib, sys
sys.modules.pop("lab8_kit", None)
import lab8_kit as K
importlib.reload(K)

print(f"lab8_kit.py written: {len(kit_source.splitlines())} lines")
print("provider:", K.PROVIDER, "| model:", K.MODEL)
print("corpus  :", list(K.CORPUS))
print("briefs  :", [b.brief_id for b in K.BRIEFS])

lab8_kit.py written: 420 lines
provider: ollama | model: qwen2.5:7b
corpus  : ['SUP-2401', 'SUP-2402', 'FIN-1180', 'OPS-0907', 'LEG-0455']
briefs  : ['B1', 'B2', 'B3', 'B4']


In [ ]:
import json, re, time
from typing import Any
from pydantic import ValidationError
from lab8_kit import (AgentRun, Analysis, Brief, CrewTrace, Memo, Message,
                      Research, Role, ROLES, Verdict)

client = K.make_client()
r = client.chat.completions.create(
    model=K.MODEL, temperature=0, max_tokens=40,
    messages=[{"role": "user", "content": "Reply with the single word: ready"}])
print("model says:", r.choices[0].message.content.strip())
print("tokens    :", r.usage.total_tokens)

model says: Ready
tokens    : 38



## Part 1 — The world (given)

Five documents, four briefs. Small and closed on purpose: every failure you
see is one you caused, not noise from a large corpus.

In [ ]:
for b in K.BRIEFS:
    print(f"{b.brief_id}  {b.question}")
    print(f"     gold docs: {sorted(b.gold_docs) or '(none — see the trap)'}")
    print(f"     {b.trap}\n")

B1  Why did customer late-delivery credits rise in Q2, and what can we recover from the supplier?
     gold docs: ['FIN-1180', 'LEG-0455', 'OPS-0907', 'SUP-2402']
     The answer spans four documents and the LEGAL constraint reverses the obvious conclusion: you cannot recover the credits. A crew that splits research from analysis often loses LEG-0455 at the handoff — inter-agent misalignment, and the commonest MAST family after specification.

B2  Summarise Delta Logistics' on-time performance across Q1 and Q2.
     gold docs: ['SUP-2401', 'SUP-2402']
     Two documents, two numbers, no reasoning. A single agent should match or beat the crew here, and the token ratio is the point: this is the case that shows coordination costing more than it buys.

B3  Should we terminate the Delta contract?
     gold docs: ['LEG-0455', 'OPS-0907', 'SUP-2402']
     There is no correct answer, only a defensible one — and the evidence cuts both ways: Delta's decline has a stated cause and termination nee

### The role contracts

Four typed contracts. Note `Finding.doc_id` — a claim without a source
cannot survive a handoff intact, and **losing the source is how a crew
fabricates**.

In [ ]:
import inspect
print(inspect.getsource(K.Finding))
print(inspect.getsource(K.Verdict))
print("--- the researcher's system prompt ---")
print(K.SYSTEMS[K.Role.RESEARCHER][:420])

class Finding(BaseModel):
    """What the researcher hands on. Note `doc_id`: a claim without a source
    cannot survive a handoff intact, and losing the source is how a crew
    fabricates."""
    model_config = ConfigDict(extra="forbid")
    doc_id: str
    claim: str = Field(max_length=300)

class Verdict(BaseModel):
    """The critic's output. `unsupported` is the field that matters: a claim
    with no citation is the failure this whole week is about."""
    model_config = ConfigDict(extra="forbid")
    approved: bool
    unsupported: list[str] = Field(default_factory=list, max_length=6)
    missing: list[str] = Field(default_factory=list, max_length=6)

--- the researcher's system prompt ---
You are the RESEARCHER on a small analyst team.

Your only job is to retrieve evidence and report it faithfully. You do not
draw conclusions and you do not write prose for a reader.

Use search(query) results only. Every finding MUST carry the doc_id it came
from. If the corpus does not answ

### The agent caller (given)

One function, one agent turn, and it **measures that agent separately**. You
cannot argue about coordination overhead from a single total.

In [ ]:
JSON_OBJ = re.compile(r"\{.*\}", re.S)
REPAIRS = {"unfenced": 0, "retries": 0, "gave_up": 0}


def call_agent(role_system, user, contract, label, tries=3):
    """Returns (parsed | None, AgentRun carrying that agent's cost)."""
    run = AgentRun(role=label)
    t0, prompt = time.time(), user
    for _ in range(tries):
        r = client.chat.completions.create(
            model=K.MODEL, temperature=0, max_tokens=800,
            messages=[{"role": "system", "content": role_system},
                      {"role": "user", "content": prompt}])
        run.tokens += r.usage.total_tokens
        raw = r.choices[0].message.content or ""
        obj = None
        try:
            obj = json.loads(raw)                      # unrepaired, counted
        except json.JSONDecodeError:
            m = JSON_OBJ.search(raw)
            if m:
                REPAIRS["unfenced"] += 1
                try: obj = json.loads(m.group(0))
                except json.JSONDecodeError: obj = None
        if obj is not None:
            try:
                parsed = contract.model_validate(obj)
                run.seconds = time.time() - t0
                return parsed, run
            except ValidationError as exc:
                why = exc.errors()[0]["msg"]
        else:
            why = "not valid JSON"
        run.retries += 1
        REPAIRS["retries"] += 1
        prompt = (f"{user}\n\nYour previous reply was rejected: {why}. "
                  "Return ONLY the corrected JSON object.")
    REPAIRS["gave_up"] += 1
    run.ok, run.error = False, "no valid output"
    run.seconds = time.time() - t0
    return None, run


def evidence(brief, k=4):
    hits = K.search(brief.question, k=k)
    return ("\n".join(f"[{h['doc_id']}] {h['text']}" for h in hits)
            or "(the search returned nothing)")

print("caller ready")

caller ready



## Part 2 — Task 1: the baseline

One agent, one call, **all** the evidence. Note what it does not have: no
handoff, so no context is lost between roles, and no coordination tokens are
spent.

> ### 🔧 Task 1
> Retrieve with `evidence(brief)`, put everything in one prompt, call once
> against `K.SINGLE_AGENT_SYSTEM` and the `Memo` contract, and record the
> `AgentRun` on the trace.

In [ ]:
def single_agent(brief: Brief) -> CrewTrace:
    """TODO(1): one agent, one call, all the evidence."""
    raise NotImplementedError("single_agent")


tr = single_agent(K.BRIEFS[0])
print(tr.render())
print("\n", tr.memo.text if tr.memo else "(no memo)")
print("\nscore:", K.score_output(K.BRIEFS[0], tr.memo))

In [ ]:
# @title ✅ Solution — Task 1  { display-mode: "form" }


B1  [single]
   single          575 tok  243.3s  retries=0  ok
   total 575 tok  243.3s  stop: complete

 Customer late-delivery credits rose in Q2 due to a decline in on-time deliveries by Delta Logistics, particularly in the northern depot. This decline, from 97% in Q4 to 89% in Q2, led to a significant increase in late-delivery credits from £4,100 in Q1 to £9,700 in Q2. The northern depot accounted for approximately 71% of these Q2 credits. Northwind can recover only the liquidated damages specified in the contract, which is 2% per week of delay, capped at 10% of the consignment value, and cannot claim additional consequential losses such as customer credits.

score: {'coverage': 0.3333333333333333, 'grounded': 0.75, 'cited_docs': ['FIN-1180', 'SUP-2401', 'SUP-2402', 'LEG-0455'], 'hallucinated_docs': [], 'answered': True}


**Record that number.** Everything below is measured against it.


## Part 3 — Task 2: the crew

Three things matter more than the wiring:

1. **The analyst sees the findings only, never the corpus.** That is the
   specialisation the crew exists for — and the only place context can be
   lost. Do not quietly pass the corpus through.
2. **Log every handoff** as a `Message`. You cannot argue about context loss
   without measuring what crossed the boundary.
3. **Bound the critic loop.** An unapproved memo and an unbounded reviser is
   a crew that never stops.

> ### 🔧 Task 2
> Build `crew(brief, with_critic=True, max_revisions=1)`.

In [ ]:
def crew(brief: Brief, with_critic: bool = True,
         max_revisions: int = 1) -> CrewTrace:
    """TODO(2): researcher -> analyst -> writer, then a critic that may send
    it back. Log every handoff. Bound the loop."""
    raise NotImplementedError("crew")


tr = crew(K.BRIEFS[0])
print(tr.render())
print("\n", tr.memo.text if tr.memo else "(no memo)")

In [ ]:
# @title ✅ Solution — Task 2  { display-mode: "form" }


B1  [critic]
   researcher      565 tok  219.9s  retries=0  ok
   analyst         381 tok  157.1s  retries=0  ok
   writer          453 tok  167.2s  retries=0  ok
   critic          429 tok  137.7s  retries=0  ok
   writer-r1       505 tok   89.3s  retries=0  ok
   critic-r1       419 tok   56.1s  retries=0  ok
   researcher -> analyst: 418 chars
   analyst -> writer: 481 chars
   writer -> critic: 317 chars
   critic -> writer: 183 chars
   writer -> critic: 321 chars
   critic -> writer: 126 chars
   critic: approved=False unsupported=1
   total 2752 tok  827.3s  stop: capped: 1 revision(s), not approved

 Late-delivery credits to customers increased from £4,100 in Q1 to £9,700 in Q2, with 71% of Q2 credits related to consignments through the northern depot. The exact reasons for the increase in late deliveries beyond the northern depot issues and the specific actions Northwind can take from the supplier are not detailed.



## Part 4 — Task 3: measure

**Predict the table before you run it.** Which brief should the crew win?
Which should it lose? What token ratio do you expect?

> ### 🔧 Task 3
> Run both implementations over every brief and print the ledger with
> `K.compare`.

In [ ]:
def measure(briefs=None, with_critic: bool = True):
    """TODO(3): both implementations, every brief, then K.compare."""
    raise NotImplementedError("measure")


measure()

In [ ]:
# @title ✅ Solution — Task 3  { display-mode: "form" }


brief  topology      tokens   secs  coverage  grounded  notes
-------------------------------------------------------------------------
B1     single           564  125.4      67%       67%  
B1     critic          2756  507.0      33%      100%  4.9× baseline
B2     single           547  224.5     100%       67%  
B2     critic          4530 1148.1     100%       67%  8.3× baseline
B3     single           547  233.9     100%       75%  
B3     critic          4384 1177.3      50%       75%  8.0× baseline
B4     single           504  198.5       0%        0%  
B4     critic          2715  692.5       0%        0%  5.4× baseline

Reported industry figures put multi-agent at 3-10× the tokens
of a single agent for equivalent work. What did YOU measure,
and did the extra spend buy anything on the coverage column?


### Read your table

1. On which briefs did the crew win, and by how much on **coverage**?
2. What was your **token ratio**? Compare with the reported 3–10×.
3. **B2** is two documents and two numbers. What did the crew's extra spend
   buy there? Be honest.


## Part 5 — Exercises

Nothing here is scripted. Report what happened even when it is not what the
note predicts — a failure that did not occur is itself a result.

### Exercise 1 — Where the context stops

`B1` needs four documents, and `LEG-0455` reverses the obvious conclusion.
The analyst never sees the corpus.

In [ ]:
tr = crew(K.BRIEFS[0])
print(tr.render())
print("\nWHAT CROSSED EACH BOUNDARY:")
for m in tr.messages:
    print(f"\n--- {m.frm} -> {m.to} ({m.chars} chars) ---")
    print(m.payload[:400])

# Q1. Did LEG-0455 survive the researcher -> analyst handoff?
# Q2. If it did not, what did the analyst conclude — and was that
#     conclusion reasonable GIVEN WHAT IT WAS HANDED?
# Q3. Nobody lied. Which MAST family is this?

### Exercise 2 — The brief the crew cannot win

`B2` is two documents and two numbers. No reasoning required.

In [ ]:
base = single_agent(K.BRIEFS[1])
cr = crew(K.BRIEFS[1])
for lbl, t in (("baseline", base), ("crew", cr)):
    sc = K.score_output(K.BRIEFS[1], t.memo)
    print(f"{lbl:<9} {t.tokens:>6} tok  coverage {sc['coverage']:>4.0%}  "
          f"grounded {sc['grounded']:>4.0%}")
print(f"\nratio: {cr.tokens / max(base.tokens, 1):.1f}\u00d7")

# Q1. Did the crew produce a better answer? By what measure?
# Q2. If the answers are equivalent, what exactly did you buy?
# Q3. This is the case the 2026 evidence is about. Say so in the memo.

### Exercise 3 — Sycophancy

`B3` has no correct answer and the evidence cuts both ways. Watch whether
the analyst simply agrees with however the researcher framed it.

In [ ]:
tr = crew(K.BRIEFS[2])
print(tr.render())
print("\nresearcher framing:")
print(tr.messages[0].payload[:350])
print("\nanalyst conclusion:")
print(tr.messages[1].payload[:350] if len(tr.messages) > 1 else "(none)")

# Q1. Did the analyst weigh both sides, or restate the researcher?
# Q2. The critic approved it — but the critic checks GROUNDING, not
#     BALANCE. What check would catch a one-sided conclusion?
# Q3. Which MAST family?

### Exercise 4 — The brief with no answer

`B4` asks about Q3. **The corpus has no Q3 data at all.** The correct output
says so.

In [ ]:
base = single_agent(K.BRIEFS[3])
cr = crew(K.BRIEFS[3])
for lbl, t in (("baseline", base), ("crew", cr)):
    sc = K.score_output(K.BRIEFS[3], t.memo)
    print(f"\n--- {lbl} ---")
    print(t.memo.text if t.memo else "(no memo)")
    print("citations:", t.memo.citations if t.memo else [],
          "| hallucinated:", sc["hallucinated_docs"])

# Q1. Did either produce a memo anyway? Did it cite documents that do not
#     exist, or real documents that do not answer the question?
# Q2. Did the critic catch it? It checks grounding against the FINDINGS.
# Q3. Which MAST family — and what would have caught it?

### Exercise 5 — What the critic costs

Run the crew with and without it.

In [ ]:
for with_critic in (False, True):
    tot = 0
    for b in K.BRIEFS:
        tot += crew(b, with_critic=with_critic).tokens
    print(f"critic={with_critic}: {tot} tokens across four briefs")

# Q1. What did the critic add, in tokens and in seconds?
# Q2. Did it change any OUTCOME, or only the cost?
# Q3. Whether that is worth it depends on the cost of publishing something
#     wrong. That is a product decision, not an engineering one — say so.

### Stretch — give the critic the corpus

Exercise 1 showed the critic cannot catch what was never handed on, because
it checks the memo against the **findings**.

Give it the brief's required sources instead and re-run `B1`. What changes,
and what does it now cost?

In [ ]:
def coverage_critic(brief, memo) -> Verdict:
    """Stretch: a deterministic check against the brief's gold_docs.

    Hint: no model call at all. Compare memo.citations with
    brief.gold_docs and report what is missing.
    """
    raise NotImplementedError("stretch")


## Submit

- this notebook, executed
- `crew.py` — your baseline, crew and measurement harness
- your ledger table
- `decision_memo.md`

### The decision memo

1. **Did the crew beat the baseline?** On which briefs, and by how much?
   Quote coverage **and** grounding, not one number.
2. **What was your token ratio?** Compare with the reported 3–10×.
3. **Where did context get lost?** There is exactly one place it can.
4. **What did the critic buy, and what did it cost?**
5. **Classify every failure you saw** using the three MAST families.
6. **What would you actually ship — and what did this lab not tell you?**

Questions 1 and 5 carry the most marks. For question 6, note that nothing
here tests **parallelism**, which is the case multi-agent is strongest for;
that four briefs and one run each gives you no variance estimate; and that a
7B model fails differently from a frontier one.

### Before Week 9

Bring your ledger, one failure you could classify confidently, and one you
could not. Next week the agent stops talking to other agents and starts
answering to a person.